In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys; sys.path.append("/data/jerrylee/pjt/BIGFAM.v.0.1")
from BIGFAM import tools

In [52]:
source = "GS"

In [53]:
# load data
path = f"/data/jerrylee/pjt/BIGFAM.v.0.1/data/{source}/obj2"
fns = os.listdir(path)
fns = [fn for fn in fns if fn.split(".")[-1] == "X"]

df = pd.DataFrame(columns=["pheno", "X", "lower", "upper"])
for fn in fns:
    pheno = fn.split(".")[0]
    df_tmp = pd.read_csv(f"{path}/{fn}", sep='\t')
    
    df.loc[len(df)] = [pheno] + list(df_tmp[df_tmp["param"] == "X"].values[0][1:])

df.sort_values(by="lower")

,pheno,X,lower,upper
0,PR_interval,-1.590068e-01,-0.235227,-0.112746
13,ECG_Count,-1.812503e-01,-0.233144,-0.119542
26,P_duration,-1.377431e-01,-0.207673,-0.051038
17,Creatinine,-6.088497e-02,-0.105374,-0.025118
5,P_axis,-5.430396e-02,-0.096137,-0.001644
10,Creat_mgdl,-4.136117e-02,-0.083406,-0.011568
11,Sodium,-2.847986e-02,-0.071252,0.020906
31,expected,-7.002465e-03,-0.056825,0.045433
21,HDL_cholesterol,1.678201e-03,-0.019367,0.018405
30,FEV,-3.505329e-03,-0.018664,0.009537


In [54]:
df_x = df[df["X"] > 1e-6]
df_x.mean()

/tmp/ipykernel_151528/3598186973.py:2: FutureWarning: Dropping of nuisance columns in DataFrame reductions (with 'numeric_only=None') is deprecated; in a future version this will raise TypeError.  Select only valid columns before calling the reduction.
  df_x.mean()


X        0.002914
lower   -0.001148
upper    0.008740
dtype: float64

In [55]:
obj1_path = f"/data/jerrylee/pjt/BIGFAM.v.0.1/data/{source}/obj1"

pheno_fns = os.listdir(obj1_path)
phenos = [pheno_fn.split(".")[0] for pheno_fn in pheno_fns]
phenos = set(phenos)

df_bigfam = pd.DataFrame()
for pheno in phenos:
    try:
        tmp = pd.read_csv(f"{obj1_path}/{pheno}.GSW", sep='\t')
        
        tmp["param"] = ["G_BIGFAM", "S_BIGFAM", "w_BIGFAM", "mse_train", "mse_test"]
        tmp_wide = tools.long2wide(tmp)
        # tmp_sig = pd.read_csv(f"{frlog_path}/{pheno}.FRLOG_raw", sep='\t')
        # sig = _slopeSig(tmp_sig["slope"])
        tmp_wide["pheno"] = pheno
        # tmp_wide["sig"] = sig
        
        df_bigfam = pd.concat([df_bigfam, tmp_wide], axis=0)
    except:
        continue

In [56]:
sig_G = df_bigfam["lower_G_BIGFAM"] > 1e-6
sig_S = df_bigfam["lower_S_BIGFAM"] > 1e-6
df_as = df_bigfam[sig_G & sig_S]

# df_as = df_bigfam.copy()
df_as.shape[0]

14

In [57]:
df_mrg = pd.merge(df_x, df_as, on="pheno")
df_mrg.shape

(10, 19)

In [58]:
(df_mrg["X"] / df_mrg["G_BIGFAM"]).mean()

0.02958235156073517

In [59]:
df_mrg["X"] / df_mrg["G_BIGFAM"]

0    0.035378
1    0.043460
2    0.014383
3    0.005177
4    0.063941
5    0.049137
6    0.007397
7    0.003688
8    0.029533
9    0.043731
dtype: float64

In [60]:
df_mrg

,pheno,X,lower,upper,G_BIGFAM,lower_G_BIGFAM,upper_G_BIGFAM,S_BIGFAM,lower_S_BIGFAM,upper_S_BIGFAM,w_BIGFAM,lower_w_BIGFAM,upper_w_BIGFAM,mse_train,lower_mse_train,upper_mse_train,mse_test,lower_mse_test,upper_mse_test
0,avg_sys,0.003428,0.001304,0.009341,0.096884,0.095660,0.098432,0.040679,0.039775,0.041338,0.95,0.95,0.95,269.010931,228.354445,276.289480,25.664916,18.450950,66.413729
1,max_arm,0.006110,0.001741,0.013933,0.140579,0.137212,0.143020,0.065918,0.064857,0.067353,0.95,0.95,0.95,9.988221,9.204839,10.469828,1.057030,0.580508,1.991408
2,avg_hr,0.001560,-0.003185,0.008106,0.108429,0.106996,0.109977,0.071666,0.070889,0.072251,0.95,0.95,0.95,90.775196,88.552347,92.724966,10.046215,8.097858,12.275250
3,ratio,0.000669,-0.000427,0.003378,0.129182,0.126206,0.131697,0.050080,0.048974,0.051280,0.95,0.95,0.95,37.027505,35.122533,38.237864,4.054787,2.856611,5.972657
4,FVC,0.019946,-0.002935,0.049462,0.311951,0.309137,0.315210,0.104252,0.102797,0.105336,0.95,0.95,0.95,13.831270,13.284570,14.279069,1.531955,1.081874,2.078987
5,max_sys,0.004290,0.001328,0.010686,0.087310,0.085615,0.088705,0.046229,0.045346,0.047098,0.95,0.95,0.95,172.602311,167.477584,177.863598,19.059629,13.777716,24.209434
6,Total_cholesterol,0.001298,-0.000520,0.006033,0.175468,0.173421,0.177674,0.030695,0.029801,0.031550,0.95,0.95,0.95,92.888363,89.296112,95.727684,10.173596,7.343407,13.787060
7,HDL_cholesterol,0.001678,-0.019367,0.018405,0.455049,0.451846,0.457618,0.049490,0.048517,0.050764,0.95,0.95,0.95,8.589694,8.281987,8.853984,0.957657,0.692795,1.265257
8,avg_dia,0.002853,0.000750,0.008230,0.096613,0.095322,0.098204,0.051892,0.051278,0.052500,0.95,0.95,0.95,159.688269,153.473882,163.589650,17.181984,13.298067,23.456850
9,max_leg,0.007008,0.001631,0.014793,0.160252,0.157868,0.162608,0.068798,0.067854,0.069859,0.95,0.95,0.95,10.768658,10.344651,11.285904,1.211079,0.696001,1.638001
